# Robust / Uncertain-input Classification

39番Notebookの標準binary GP contractを前提に、label noiseとcandidate-time input uncertaintyを比較します。


In [ ]:
import torch

from robotorchan.models.classification import (
    ContinuousUncertainInputBinarySingleTaskGPClassifier,
    LabelNoiseBinarySingleTaskGPClassifier,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


## 1. 小さな合成データ

In [ ]:
train_X = torch.tensor([[0.05], [0.15], [0.30], [0.48], [0.52], [0.70], [0.85], [0.95]])
train_Y = (train_X.squeeze(-1) > 0.5).to(dtype=torch.double)
grid = torch.linspace(0.0, 1.0, 51).unsqueeze(-1)


## 2. Label noise と input uncertainty

Label noiseは観測過程、input uncertaintyはcandidate入力の不確かさです。

In [ ]:
label_noise_model = LabelNoiseBinarySingleTaskGPClassifier(train_X, train_Y)
uncertain_model = ContinuousUncertainInputBinarySingleTaskGPClassifier(train_X, train_Y)
print(label_noise_model.classification_metadata)
print(uncertain_model.classification_metadata)


## 3. Candidate-time input uncertainty

In [ ]:
input_std = torch.full_like(grid, 0.03)
with torch.no_grad():
    probability = uncertain_model.predict_proba_with_input_uncertainty(
        grid, input_std=input_std, num_samples=32
    )
print(probability.shape)


## 4. 使い分け

- 誤ラベルはrobust likelihood側で扱います。
- candidate Xの揺らぎはuncertain-input側で扱います。
- Student-t regression likelihoodをbinary labelへ機械的に移植しません。
- categorical input uncertaintyには専用のexact marginalization APIがあります。

## 5. 学習・候補提案・ラベル評価

両モデルを既存の変分ELBOで学習し、入力不確実性を考慮した予測確率に基づいて、未ラベル候補を選択します。候補の選択は有限プールで行います。


In [ ]:
for model in (label_noise_model, uncertain_model):
    model.train()
    model.likelihood.train()
    mll = model.make_mll()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
    for _ in range(20):
        optimizer.zero_grad()
        loss = -mll(model.model(train_X), train_Y)
        loss.backward()
        optimizer.step()
    model.eval()
    model.likelihood.eval()

candidate_pool = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
with torch.no_grad():
    probabilities = uncertain_model.predict_proba_with_input_uncertainty(
        candidate_pool,
        input_std=torch.full_like(candidate_pool, 0.03),
        num_samples=32,
    )
    positive_probability = probabilities[..., 1]
    entropy = -(
        positive_probability.clamp(1e-8, 1 - 1e-8).log() * positive_probability
        + (1 - positive_probability).clamp(1e-8, 1 - 1e-8).log()
        * (1 - positive_probability)
    )
candidate = candidate_pool[entropy.argmax()].unsqueeze(0)
observed_label = (candidate.squeeze(-1) > 0.5).long()
print("candidate:", candidate, "observed label:", observed_label)


## 初学者向け補足：何が不確かなのかを区別する

同じ「不確実性」でも、原因が異なると対策も異なります。

| 状況 | 例 | 主な対処 |
|---|---|---|
| ラベルノイズ | 検査員ごとに合否判定が揺れる | ラベル生成・観測過程の頑健化 |
| 入力不確実性 | 設定温度と実温度にずれがある | 入力の揺らぎを考慮した予測 |
| モデル不確実性 | 未測定の領域が広い | 追加観測・Active Learning |

### Notebookの読み進め方

1. 合成データの0/1ラベルを確認します。
2. どのモデルがラベルノイズを扱い、どの処理が入力の揺らぎを扱うかを追います。
3. 学習後の予測確率を比較し、入力が揺れたときに候補の評価が変わるかを確認します。
4. 候補提案後に得られるラベルは実験・測定で初めて判明することを意識します。合成データの真値を使うのはデモ用です。

**注意：** ラベルノイズへの頑健性と、未知の入力変動への頑健性は同義ではありません。例の少ないepochや小さな合成データから、実問題での優劣を結論づけないでください。